# 06 — LangChain Basics & RAG with a Text File

**Day 2 | Data Engineering & AI Bootcamp**

This notebook teaches the complete RAG (Retrieval Augmented Generation) pipeline
using **LangChain** — the most widely used framework for building LLM applications.

We go from basics → full RAG system that answers questions from a real product catalog file.

**Flow:**
1. LangChain basics — Prompts, LLMs, Chains
2. Load & chunk a text file
3. Embed chunks with OpenAI
4. Store in Chroma vector DB
5. Retrieve + Generate answers (RAG)
6. Conversation memory

```
pip install langchain langchain-openai langchain-community chromadb tiktoken
export OPENAI_API_KEY='sk-...'
```


In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# LangChain core
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain document handling
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# OpenAI (embeddings + LLM)
try:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    HAS_OPENAI = bool(os.getenv('OPENAI_API_KEY', ''))
except ImportError:
    HAS_OPENAI = False

print(f'OpenAI API key set: {HAS_OPENAI}')
print('Setup complete ✓')


## 1. LangChain Basics — Prompts

A **Prompt Template** is a reusable text template with variables.
Instead of writing the same prompt string every time, you define it once with `{placeholders}`
and fill them in at call time.

> **Why?** In production, the same prompt structure is used thousands of times with different inputs.
> Templates make prompts version-controlled, testable, and reusable.


In [ ]:
# ── PromptTemplate — basic string prompt ──────────────────────────
product_prompt = PromptTemplate.from_template(
    "You are a product expert. Summarise this product in one sentence:\n\n{product_description}"
)

compare_prompt = PromptTemplate.from_template(
    "Compare {product_a} and {product_b} on: price, performance, and best use case."
)

# Fill in variables
filled = product_prompt.format(product_description="A 65-inch OLED TV with perfect blacks and Dolby Atmos.")
print("Filled prompt:")
print(filled)
print()

# ── ChatPromptTemplate — for chat models (system + human messages) ─
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful tech sales assistant. Be concise and friendly."),
    ("human",  "What is the best laptop for {use_case} under {budget}?"),
])

msgs = chat_prompt.format_messages(use_case="machine learning", budget="₹2 lakh")
print("Chat prompt messages:")
for m in msgs:
    print(f"  [{m.__class__.__name__}] {m.content}")


## 2. LangChain Basics — Chains (LCEL)

**LCEL** (LangChain Expression Language) uses the `|` pipe operator to connect components.
Each component takes an input and produces an output — they chain together like Unix pipes.

```
prompt | llm | output_parser
```

This is declarative: you describe what should happen, LangChain handles execution,
streaming, batching, and error handling.

> **Real world:** A Databricks LLM pipeline at Hexaware uses LangChain chains to route
> customer queries: classify intent → select prompt template → call LLM → parse response.


In [ ]:
if HAS_OPENAI:
    # ── Simple chain: prompt | LLM | parser ───────────────────────
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    simple_chain = chat_prompt | llm | StrOutputParser()

    response = simple_chain.invoke({
        "use_case": "data science and machine learning",
        "budget": "₹2 lakh"
    })
    print("Chain response:")
    print(response)
    print()

    # ── Multi-step chain ───────────────────────────────────────────
    summarise_prompt = ChatPromptTemplate.from_messages([
        ("system", "You summarise product descriptions in exactly one sentence."),
        ("human",  "{description}"),
    ])

    bullet_prompt = ChatPromptTemplate.from_messages([
        ("system", "Convert this summary into 3 bullet points."),
        ("human",  "{summary}"),
    ])

    # Chain: summarise → then convert to bullets
    chain1 = summarise_prompt | llm | StrOutputParser()
    chain2 = bullet_prompt    | llm | StrOutputParser()

    description = ("MacBook Pro 14 with M3 Pro chip, 18GB RAM, Liquid Retina XDR display, "
                   "18-hour battery, Thunderbolt 4, best for ML engineers and data scientists.")

    summary = chain1.invoke({"description": description})
    bullets = chain2.invoke({"summary": summary})

    print("Summary:", summary)
    print()
    print("Bullets:")
    print(bullets)

else:
    print("No OPENAI_API_KEY — here is what a chain looks like:")
    print()
    print("  llm    = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)")
    print("  chain  = prompt | llm | StrOutputParser()")
    print("  result = chain.invoke({'use_case': 'data science', 'budget': '₹2 lakh'})")
    print()
    print("  Output → 'The MacBook Pro M3 is ideal for data science under ₹2 lakh...'")
    print()
    print("LCEL pipe operator | connects: Prompt → LLM → OutputParser")
    print("Each step takes the previous step's output as input.")


## 3. Load & Chunk a Text File

Before we can search documents, we need to:
1. **Load** the file into LangChain `Document` objects
2. **Chunk** the content into small segments (each chunk gets one embedding)

**Why chunk?** Embedding models have a token limit (~512 tokens).
A 10-page document must be split into ~50 chunks of ~200 words.
Each chunk is independently embedded and indexed.

**RecursiveCharacterTextSplitter** tries to split on paragraphs → sentences → words,
in that order, to preserve as much context as possible per chunk.


In [ ]:
from pathlib import Path

DATA_FILE = Path('../data/product-data.txt')

# ── Load using LangChain TextLoader ───────────────────────────────
loader = TextLoader(str(DATA_FILE), encoding='utf-8')
raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} document(s)")
print(f"Total characters: {len(raw_docs[0].page_content):,}")
print(f"Source metadata: {raw_docs[0].metadata}")
print()
print("First 300 characters of document:")
print(raw_docs[0].page_content[:300])


In [ ]:
# ── Split into chunks ─────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 400,   # max chars per chunk (~100 words)
    chunk_overlap = 80,    # overlap so context isn't lost at boundaries
    separators    = ["\n\n", "\n", ". ", " "],  # try paragraph > line > sentence
)

chunks = splitter.split_documents(raw_docs)

print(f"Total chunks:          {len(chunks)}")
print(f"Avg chunk length:      {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"Min / Max chunk:       {min(len(c.page_content) for c in chunks)} / {max(len(c.page_content) for c in chunks)} chars")
print()
print("Sample chunk #3:")
print("-" * 50)
print(chunks[3].page_content)
print("-" * 50)
print(f"Metadata: {chunks[3].metadata}")


## 4. Embed Chunks & Store in Chroma

Each chunk is converted to an embedding vector and stored in ChromaDB.
This is the **indexing phase** of RAG — done once, then reused for many queries.

**OpenAI `text-embedding-3-small`:**
- 1536 dimensions
- Great for retrieval tasks
- Cost: $0.02 per million tokens (~50,000 product chunks for $0.001)

ChromaDB stores: the text, its vector embedding, and metadata (source file, chunk index).


In [ ]:
if HAS_OPENAI:
    # ── Embed and store in Chroma (in-memory) ─────────────────────
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    vectorstore = Chroma.from_documents(
        documents  = chunks,
        embedding  = embeddings,
        collection_name = "product_catalog",
    )

    print(f"Indexed {vectorstore._collection.count()} chunks in ChromaDB")
    print()

    # ── Test retrieval ─────────────────────────────────────────────
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    test_query = "best laptop for machine learning"
    results = retriever.invoke(test_query)

    print(f"Query: '{test_query}'")
    print(f"Retrieved {len(results)} chunks:")
    for i, doc in enumerate(results, 1):
        print(f"\n  Chunk {i}:")
        print(f"  {doc.page_content[:200]}...")

else:
    print("No OPENAI_API_KEY — here is the indexing code:")
    print()
    print("  embeddings  = OpenAIEmbeddings(model='text-embedding-3-small')")
    print("  vectorstore = Chroma.from_documents(chunks, embeddings)")
    print("  retriever   = vectorstore.as_retriever(search_kwargs={'k': 3})")
    print()
    print("  # Each call to from_documents:")
    print("  #   1. Calls OpenAI API to embed each chunk")
    print("  #   2. Stores (text, vector, metadata) in ChromaDB")
    print("  #   3. HNSW index built automatically")

    # ── Demo with local embeddings (no API key needed) ─────────────
    print()
    print("Running demo with local sentence-transformers instead:")
    from langchain_community.embeddings import HuggingFaceEmbeddings

    local_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(
        documents       = chunks,
        embedding       = local_embeddings,
        collection_name = "product_catalog_local",
    )
    print(f"Indexed {vectorstore._collection.count()} chunks with local model")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


## 5. RAG Pipeline — Retrieve + Generate

This is where it all comes together:

```
User question
      ↓
  Embed question  (same embedding model as indexing)
      ↓
  Retrieve top-3 chunks  (HNSW similarity search in Chroma)
      ↓
  Build prompt: system + retrieved context + user question
      ↓
  Call LLM (GPT-3.5 / GPT-4)
      ↓
  Return grounded answer with source citations
```

The LLM **cannot hallucinate** product specs because we explicitly instruct it to
answer only from the provided context.


In [ ]:
if HAS_OPENAI:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
else:
    # Fallback: a mock LLM that shows what the answer would look like
    from langchain_core.language_models.fake import FakeListChatModel
    llm = FakeListChatModel(responses=[
        "Based on the product catalog, the MacBook Pro 14 M3 Pro (₹1,99,900) is the best laptop "
        "for machine learning. It features the M3 Pro chip, 18GB unified memory, and runs Python, "
        "Jupyter, and Databricks CLI flawlessly. The battery lasts 18 hours — no power anxiety.",
        "For noise cancellation, Sony WH-1000XM5 (₹29,990) uses 8 microphones and dual processors "
        "for industry-leading ANC. The Apple AirPods Pro 2 (₹24,900) is better if you use iPhone "
        "for its tighter ecosystem integration and Spatial Audio.",
        "For gaming under ₹1.5 lakh: ASUS ROG Zephyrus G14 (₹1,39,990) with AMD Ryzen 9 8945HS "
        "and RX 7700S. 144Hz display, 32GB RAM, MUX Switch for full GPU performance.",
    ])

# ── RAG prompt template ────────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a knowledgeable product advisor for TechStore India. "
     "Answer the customer's question using ONLY the product information provided below. "
     "If the answer is not in the context, say 'I don't have that information in our catalog.' "
     "Always mention the product name and price when recommending.\n\n"
     "Product Context:\n{context}"),
    ("human", "{input}"),
])

# ── Build the RAG chain ────────────────────────────────────────────
combine_docs_chain = create_stuff_documents_chain(llm, rag_prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# ── Ask questions ──────────────────────────────────────────────────
questions = [
    "What is the best laptop for machine learning and data science?",
    "Which headphones have the best noise cancellation?",
    "I want a gaming laptop under ₹1.5 lakh. What do you recommend?",
]

for q in questions:
    print(f"Q: {q}")
    result = rag_chain.invoke({"input": q})
    print(f"A: {result['answer']}")
    print(f"   (Retrieved from {len(result['context'])} chunks)")
    print()


## 6. Inspect What Was Retrieved

Debugging RAG = understanding what context was passed to the LLM.
If the answer is wrong, check:
1. Was the right chunk retrieved? (retrieval problem)
2. Was the chunk too short/long? (chunking problem)
3. Did the LLM ignore the context? (prompt problem)


In [ ]:
query = "What is the price of Sony WH-1000XM5 headphones?"

# Get retrieved docs with scores
docs_with_scores = vectorstore.similarity_search_with_score(query, k=4)

print(f"Query: '{query}'")
print(f"\nTop-4 retrieved chunks (lower score = more similar in Chroma L2):")
print("=" * 70)
for doc, score in docs_with_scores:
    print(f"Score: {score:.4f}")
    print(doc.page_content[:300])
    print("-" * 70)


## 7. Conversation Memory

A stateless RAG chain forgets the previous question on every call.
**Memory** adds the conversation history to the prompt so the LLM can answer
follow-up questions correctly.

```
User:  "What's a good laptop for ML?"      → LLM answers MacBook Pro
User:  "What's the battery life?"          → Without memory: "Battery life of what?"
                                           → With memory:    "MacBook Pro M3 has 18-hour battery."
```

LangChain stores messages in a `ConversationBufferMemory` and appends them to the prompt.


In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

if HAS_OPENAI:
    llm_chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
else:
    llm_chat = FakeListChatModel(responses=[
        "The MacBook Pro 14 M3 Pro (₹1,99,900) is ideal for machine learning "
        "with its M3 Pro chip, 18GB unified memory, and 18-hour battery life.",
        "The MacBook Pro M3 Pro has an 18-hour battery life — more than enough for a full workday.",
        "Yes, both MacBook Pro and Dell XPS 15 come with Thunderbolt 4 ports.",
    ])

memory = ConversationBufferMemory(
    memory_key          = "chat_history",
    return_messages     = True,
    output_key          = "answer",
)

conv_chain = ConversationalRetrievalChain.from_llm(
    llm       = llm_chat,
    retriever = retriever,
    memory    = memory,
    verbose   = False,
)

# ── Multi-turn conversation ────────────────────────────────────────
conversation = [
    "What is the best laptop for machine learning?",
    "What is the battery life of that laptop?",           # follow-up — needs memory
    "Does it have Thunderbolt 4 ports?",                   # another follow-up
]

print("=== CONVERSATION WITH MEMORY ===")
for turn, question in enumerate(conversation, 1):
    result = conv_chain.invoke({"question": question})
    print(f"Turn {turn}")
    print(f"  Q: {question}")
    print(f"  A: {result['answer']}")
    print()

print(f"Memory buffer has {len(memory.chat_memory.messages)} messages stored")


## Summary — What You Built

```
product-data.txt
      ↓  TextLoader
LangChain Documents
      ↓  RecursiveCharacterTextSplitter (400 chars, 80 overlap)
Chunks (~30 chunks)
      ↓  OpenAIEmbeddings text-embedding-3-small (1536 dims)
ChromaDB vector store (HNSW index)
      ↓  retriever.invoke(question)
Top-3 relevant chunks
      ↓  create_stuff_documents_chain (inject context into prompt)
ChatGPT / GPT-4
      ↓
Grounded answer with product names + prices
```

**Key LangChain concepts covered:**
| Component | What it does |
|---|---|
| `PromptTemplate` | Reusable prompt with `{variables}` |
| `ChatPromptTemplate` | System + Human message structure |
| `LCEL \|` | Pipe operator connecting components |
| `TextLoader` | Load `.txt`, `.pdf`, `.csv` files |
| `RecursiveCharacterTextSplitter` | Smart chunking |
| `OpenAIEmbeddings` | Text → 1536-dim vector via API |
| `Chroma.from_documents` | Index + store all chunks |
| `create_retrieval_chain` | Full RAG pipeline in 2 lines |
| `ConversationBufferMemory` | Multi-turn memory |

**Next steps:** Swap `Chroma` for **Databricks Vector Search**, swap `ChatOpenAI` for
**Databricks Model Serving** (Llama 3 / DBRX) — same LangChain code, enterprise scale.
